# Phase 2 — Final fixed-checkpoint evaluation

Use the same Kaggle session that completed stabilization. This notebook performs
no training. It rechecks validation against recorded scores, then evaluates source
holdout and target query at threshold 0.5. Existing results/checkpoints are retained.
If the session reset, attach the saved phase2_stabilization_results.zip as Kaggle
input; the next cell can restore it. Enable Internet if raw data needs restoration.


In [1]:
import os,sys,json,zipfile,subprocess,hashlib
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
project=Path('/kaggle/working/road_crack_phase2_v2')
if not (project/'artifacts/phase2/unet/checkpoints/best.json').exists():
    candidates=list(Path('/kaggle/input').rglob('phase2_stabilization_results.zip'))
    local=Path('/kaggle/working/phase2_stabilization_results.zip')
    if local.exists(): candidates.insert(0,local)
    assert candidates, 'Attach your phase2_stabilization_results.zip backup as Kaggle input first'
    archive=candidates[0]
    assert hashlib.sha256(archive.read_bytes()).hexdigest()=='578ea01fb8389abd0904f63ed621192bc6708b6d76cac8ca9c2840e01d1a5040', 'Unexpected backup: verify before restoring'
    project.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            path=(project/name).resolve()
            assert path.is_relative_to(project.resolve())
            if path.exists(): assert path.read_bytes()==z.read(name), 'Existing file differs; preserve it'
        z.extractall(project)
os.chdir(project)
(project/'scripts/evaluate_frozen_baselines.py').write_text('"""Final baseline evaluation: fixed validation-best weights and threshold, no tuning."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport torch\nfrom src.atomic import write_json\nfrom src.dataset import CrackDataset,verify_files\nfrom src.models import SegmentationUNet\nfrom scripts.train_baseline import evaluate\n\n\ndef run_evaluation(project,data_root,output):\n    project=Path(project); output=Path(output)\n    if output.exists(): raise ValueError(\'Evaluation output exists; preserve it and use a new directory\')\n    manifest_path=project/\'data/manifests/pilot_v1.json\'\n    manifest_bytes=manifest_path.read_bytes()\n    manifest_hash=hashlib.sha256(manifest_bytes).hexdigest()\n    if manifest_hash!=manifest_path.with_suffix(\'.json.sha256\').read_text().split()[0]:\n        raise ValueError(\'Manifest checksum mismatch\')\n    manifest=json.loads(manifest_bytes)\n    verify_files(manifest[\'records\'],data_root)\n    if not torch.cuda.is_available(): raise RuntimeError(\'Use the verified Kaggle GPU runtime\')\n    output.mkdir(parents=True)\n    summary={}\n    for architecture in (\'unet\',\'r2au\'):\n        run=project/\'artifacts/phase2\'/architecture\n        pointer=json.loads((run/\'checkpoints/best.json\').read_text())\n        path=(run/\'checkpoints\'/pointer[\'file\']).resolve()\n        if path.parent!=(run/\'checkpoints\').resolve(): raise ValueError(\'Unsafe checkpoint path\')\n        if hashlib.sha256(path.read_bytes()).hexdigest()!=pointer[\'sha256\']:\n            raise ValueError(\'Checkpoint checksum mismatch\')\n        # Complete recovery files use pickle. Only load the trusted local run.\n        state=torch.load(path,map_location=\'cpu\',weights_only=False)\n        if state[\'manifest_hash\']!=manifest_hash: raise ValueError(\'Checkpoint uses different split\')\n        config=state[\'config\']\n        device=torch.device(\'cuda\')\n        model=SegmentationUNet(architecture,config[\'base_channels\'],config.get(\'normalization\',\'none\')).to(device)\n        model.load_state_dict(state[\'model\'])\n        validation=CrackDataset(manifest,data_root,\'source_validation\',config[\'image_size\'])\n        replay,_=evaluate(model,validation,device,config[\'batch_size\'])\n        expected=json.loads((run/\'evaluation.json\').read_text())[\'results\'][\'source_validation\']\n        for key in (\'dice\',\'iou\',\'precision\',\'recall\'):\n            if abs(replay[key]-expected[key])>1e-5:\n                raise ValueError(f\'{architecture} validation replay mismatch: {key}; no query evaluation performed for this model\')\n        results={}\n        for split in (\'source_holdout\',\'target_query\'):\n            dataset=CrackDataset(manifest,data_root,split,config[\'image_size\'])\n            metrics,rows=evaluate(model,dataset,device,config[\'batch_size\'])\n            results[split]=metrics\n            write_json(output/(architecture+\'_\'+split+\'_per_image.json\'),rows)\n        summary[architecture]={\'checkpoint_sha256\':pointer[\'sha256\'],\'best_epoch\':pointer[\'epoch\'],\n            \'validation_replay\':replay,\'threshold\':.5,\'results\':results}\n        write_json(output/\'evaluation.json\',{\'manifest_sha256\':manifest_hash,\'models\':summary,\n            \'scope\':\'Fixed stabilized baselines; no training or threshold tuning; exploratory pilot query previously used in v1\'})\n        print(json.dumps({architecture:summary[architecture]},indent=2),flush=True)\n    return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--project\',type=Path,required=True)\n    p.add_argument(\'--data-root\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(); run_evaluation(a.project,a.data_root,a.output)\n')

restore_report=project/'artifacts/final_eval_dataset_restore.json'
subprocess.run([sys.executable,'scripts/restore_datasets.py','--manifest','data/manifests/pilot_v1.json',
    '--sources','configs/data_sources.json','--destination','/kaggle/working/road_crack_phase1/data/raw',
    '--report',str(restore_report)],check=True)
data_root=Path(json.loads(restore_report.read_text())['data_root'])


Reused existing data; all image/mask checksums passed: /kaggle/working/road_crack_phase1/data/raw


In [2]:
from datetime import datetime,timezone
output=project/'artifacts'/('final_evaluation_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run([sys.executable,'scripts/evaluate_frozen_baselines.py','--project',str(project),
    '--data-root',str(data_root),'--output',str(output)],check=True)
print((output/'evaluation.json').read_text())


{
  "unet": {
    "checkpoint_sha256": "ba711be7f8581815a1329202536cbb95f0e1374c989c5ec2cfbc83104f2469a8",
    "best_epoch": 87,
    "validation_replay": {
      "loss": 0.34095870653788246,
      "dice": 0.8408725599447886,
      "iou": 0.7373777742187182,
      "precision": 0.8036070277293523,
      "recall": 0.9057885736227036,
      "predicted_foreground_fraction": 0.030782063802083332,
      "truth_foreground_fraction": 0.028463999430338543,
      "n": 60
    },
    "threshold": 0.5,
    "results": {
      "source_holdout": {
        "loss": 0.32370564907411986,
        "dice": 0.8132289150721916,
        "iou": 0.7044373315912259,
        "precision": 0.8654441602119414,
        "recall": 0.806187402284095,
        "predicted_foreground_fraction": 0.03772455529321598,
        "truth_foreground_fraction": 0.04327650113573557,
        "n": 237
      },
      "target_query": {
        "loss": 0.7937848335818241,
        "dice": 0.24603936722575637,
        "iou": 0.15751775848483177

In [3]:
backup=Path('/kaggle/working/phase2_final_evaluation.zip')
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(output).as_posix())
    z.write(project/'scripts/evaluate_frozen_baselines.py','evaluate_frozen_baselines.py')
digest=hashlib.sha256(backup.read_bytes()).hexdigest()
backup.with_suffix('.zip.sha256').write_text(digest+'  '+backup.name+'\n')
print('Download through Kaggle Files:',backup)
print('SHA256:',digest)


Download through Kaggle Files: /kaggle/working/phase2_final_evaluation.zip
SHA256: e14e1e4f611cdb953a7d4132f37fd742c3c4c5bf6e5bc8b896f6c04dced5b65c
